## Anterior/posterior and medial/lateral gene pathways

**Overview**

In this tutorial, we will assess pathways involved in the anterior/posterior and medial/lateral axes of the hippocmapus

**0 - Import libraries**

In [6]:
import os
import sys

import hippomaps as hm
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import seaborn as sns

from brainspace.mesh.mesh_io import read_surface
from sklearn.metrics import r2_score
from sklearn.cross_decomposition import PLSRegression
from sklearn.discriminant_analysis import StandardScaler

sys.path.append(os.path.abspath(".."))
import hippogenes as hg

**1 - Load gene expression data**

In [2]:
expression = hg.load_expression(data_dir=".")

**2 - Load geodesic anterior-posterior and medial-lateral maps**

In [4]:
hm_path = os.path.dirname(hm.__file__)

surface = read_surface(os.path.join(hm_path, "resources", "canonical_surfs", "tpl-avg_space-unfold_den-0p5mm_label-hipp_midthickness.surf.gii"))

ap_axis = surface.Points[:,0]
ml_axis = surface.Points[:,1]

**3 - Partial least square analyses (PLS)**

In [ ]:

def pls(map, expression, n_components=5):
    
    X = StandardScaler().fit_transform(expression)
    Y = StandardScaler().fit_transform(map.reshape(-1, 1))
    
    model = PLSRegression(n_components=n_components)
    model.fit(X, Y)
    
    XL = model.x_loadings_
    YL = model.y_loadings_
    XS = model.x_scores_
    YS = model.y_scores_
    
    r2 = np.zeros(n_components)
    for i in range(n_components):
        Y_pred = np.dot(XS[:, i].reshape(-1, 1), YL[:,i].reshape(1, -1))
        r2[i] = r2_score(Y, Y_pred)
        
        if np.corrcoef(XS[:, i], Y[:, 0])[0, 1] < 0:
            XL[:,i] *= - 1
            XS[:, i] *= -1
            YL[:, i] *= -1
            YS[:, i] *= -1
            
    return XL, YL, XS, YS, r2


def pls_perm(map, expression, n_components=5, n_perm=1000):
    
    _, _, _, _, r2_obs = pls(map, expression, n_components)
    
    null = np.zeros((n_perm, n_components))
    r2_perm = np.zeros((n_perm, n_components))
    
    for i in range(n_perm):
        permuted_map = np.random.permutation(map)
        _, _, _, _, r2 = pls(permuted_map, expression, n_components)
        r2_perm[i] = r2
        
    return r2_perm

In [ ]:
imgfix = np.random.rand(10)
imgperm = np.concatenate([np.random.permutation(imgfix) for i in range(5)])

np.corrcoef(np.concatenate((imgfix.reshape([-1,1]),imgperm.reshape([-1,5])), axis=1).T)

array([[ 1.        ,  0.24629413, -0.46763489,  0.02634442,  0.38711691,
        -0.20583828],
       [ 0.24629413,  1.        , -0.49097036, -0.06214237, -0.38736471,
        -0.23511864],
       [-0.46763489, -0.49097036,  1.        ,  0.37167663, -0.24788693,
         0.31650861],
       [ 0.02634442, -0.06214237,  0.37167663,  1.        ,  0.26145111,
        -0.27226587],
       [ 0.38711691, -0.38736471, -0.24788693,  0.26145111,  1.        ,
         0.05736392],
       [-0.20583828, -0.23511864,  0.31650861, -0.27226587,  0.05736392,
         1.        ]])

In [11]:
np.concatenate((imgfix.reshape([-1,1]),imgperm.reshape([-1,5])), axis=1).shape

(10, 6)